In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-large-en-v1.5")
ds = load_dataset("hugginglearners/netflix-shows")

device = "cuda"

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
dtrain = ds["train"]

In [ ]:
print(dtrain[0])

{'show_id': 's1', 'type': 'Movie', 'title': 'Dick Johnson Is Dead', 'director': 'Kirsten Johnson', 'cast': None, 'country': 'United States', 'date_added': 'September 25, 2021', 'release_year': 2020, 'rating': 'PG-13', 'duration': '90 min', 'listed_in': 'Documentaries', 'description': 'As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.'}


In [ ]:
def format_show(row):
    def clean(val):
        return str(val).strip() if val is not None else ""

    parts = []
    if row.get('title'):       parts.append(f"Title: {clean(row['title'])}")
    if row.get('type'):        parts.append(f"Type: {clean(row['type'])}")
    if row.get('listed_in'):   parts.append(f"Genre: {clean(row['listed_in'])}")
    if row.get('description'): parts.append(f"Description: {clean(row['description'])}")
    if row.get('director'):    parts.append(f"Director: {clean(row['director'])}")
    if row.get('cast'):        parts.append(f"Cast: {clean(row['cast'])}")
    if row.get('country'):     parts.append(f"Country: {clean(row['country'])}")
    if row.get('release_year'):parts.append(f"Release Year: {clean(row['release_year'])}")
    if row.get('rating'):      parts.append(f"Rating: {clean(row['rating'])}")
    if row.get('duration'):    parts.append(f"Duration: {clean(row['duration'])}")

    return ". ".join(parts)

texts = [format_show(row) for row in dtrain]

In [ ]:
texts[0]

'Title: Dick Johnson Is Dead. Type: Movie. Genre: Documentaries. Description: As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.. Director: Kirsten Johnson. Country: United States. Release Year: 2020. Rating: PG-13. Duration: 90 min'

In [ ]:
instruction = "Represent this movie for semantic search:"
encodings = model.encode(
    texts,
    prompt=instruction,
    normalize_embeddings=True,
    batch_size=512,
    show_progress_bar=True,
    device = device
)

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

In [ ]:
from google.colab import drive
import numpy as np

# Mount drive
drive.mount('/content/drive')

# Save
save_path = "/content/drive/MyDrive/moviesembeddings.npy"
np.save(save_path, encodings)
print(f"Saved to {save_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to /content/drive/MyDrive/moviesembeddings.npy


In [ ]:
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)

    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)

    if norm_a == 0 or norm_b == 0:
        return 0.0  # avoid division by zero

    return dot_product / (norm_a * norm_b)

In [ ]:
def find_recommendations(Movie_emd, threshold=0.60):
  for i in range(0, dtrain.num_rows,1):
    score = cosine_similarity(Movie_emd, encodings[i])

    if score >= threshold:
      reccomended_movies['movie_idx'].append(i)
      reccomended_movies['score'].append(score)


In [ ]:
dtest = {}
dtest['title'] = input('Name: ')
dtest['type'] = input('Type: ')
dtest['listed-in'] = input('Genre: ')
dtest['description'] = input('Description: ')
dtest['director'] = input('Director: ')
dtest['cast'] = input('Cast: ')
dtest['country'] = input('Country: ')
dtest['release_year'] = input('Release Year: ')
dtest['rating'] = input('Rating: ')
dtest['duration'] = input('Duration: ')

Name: The Hangover
Type: Movie
Genre: Comedy
Description: Three friends wake up in Las Vegas after a wild bachelor party with no memory of the night before and must retrace their steps to find their missing friend.
Director: Todd Phillips
Cast: Bradley Cooper, Ed Helms, Zach Galifianakis, Justin Bartha
Country: United States
Release Year: 2009
Rating: R
Duration: 100 min


In [ ]:
dtest

{'title': 'The Hangover',
 'type': 'Movie',
 'listed-in': 'Comedy',
 'description': 'Three friends wake up in Las Vegas after a wild bachelor party with no memory of the night before and must retrace their steps to find their missing friend.',
 'director': 'Todd Phillips',
 'cast': 'Bradley Cooper, Ed Helms, Zach Galifianakis, Justin Bartha',
 'country': 'United States',
 'release_year': '2009',
 'rating': 'R',
 'duration': '100 min'}

In [ ]:
instruction = "Represent this movie for semantic search:"

watched = format_show(dtest)
watched_embeddings = model.encode(
    watched,
    prompt=instruction,
    normalize_embeddings=True,
    batch_size=1,
    show_progress_bar=True
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
watched_embeddings

array([ 0.04537221, -0.00116067,  0.02844682, ..., -0.02828409,
        0.01491177, -0.03383874], dtype=float32)

In [ ]:
scores = []

In [ ]:
reccomended_movies = {'movie_idx':[], 'score': []}
find_recommendations(watched_embeddings, threshold=0.70)
reccomended_movies = sorted(zip(reccomended_movies['movie_idx'],reccomended_movies['score']), key=lambda x: x[1], reverse=True)
print(reccomended_movies)

[(8386, np.float32(0.74825585)), (3560, np.float32(0.7050418)), (4115, np.float32(0.70069313)), (930, np.float32(0.6834115)), (2809, np.float32(0.67892563)), (796, np.float32(0.6738177)), (3073, np.float32(0.67379665)), (4965, np.float32(0.667191)), (4970, np.float32(0.6565494)), (5127, np.float32(0.6550595)), (1333, np.float32(0.6540711)), (4688, np.float32(0.6529858)), (5880, np.float32(0.650704))]


In [ ]:
reccomended_movies[0]

(8386, np.float32(0.74825585))

In [ ]:
print('Recommended Movies:' + '\n')

for i in range(0, len(reccomended_movies)):
  index = reccomended_movies[i][0]
  score = reccomended_movies[i][1]
  print(f"{i+1}. {dtrain[index]['title']} [{dtrain[index]['listed_in']}] [{dtrain[index]['release_year']}] score: {score}")
  index += 1

Recommended Movies:

1. The Last Hangover [Comedies, International Movies] [2018] score: 0.7482558488845825
2. Drink Drank Drunk [Comedies, International Movies] [2016] score: 0.7050418257713318
3. Ken Jeong: You Complete Me, Ho [Stand-Up Comedy] [2019] score: 0.7006931304931641
4. Due Date [Action & Adventure, Comedies] [2010] score: 0.6834114789962769
5. Search Party [Comedies] [2014] score: 0.678925633430481
6. Hostel: Part III [Cult Movies, Horror Movies] [2011] score: 0.6738176941871643
7. Road Trip: Beer Pong [Comedies] [2009] score: 0.6737966537475586
8. Trailer Park Boys: Countdown to Liquor Day [Comedies, Cult Movies] [2009] score: 0.6671910285949707
9. Game Over, Man! [Action & Adventure, Comedies] [2018] score: 0.6565493941307068
10. Judd Apatow: The Return [Stand-Up Comedy] [2017] score: 0.6550595164299011
11. War Dogs [Comedies, Dramas] [2016] score: 0.6540710926055908
12. The After Party [Comedies, Music & Musicals] [2018] score: 0.6529858112335205
13. Trailer Park Boys: 